# Priors Review & Diagnostics — Dense Visualization Pack

This notebook loads the outputs written by `results/priors/tables/*.csv` and produces a **dense, 10+ plot** diagnostic pack:

1. **Priors distributions** — histograms for μ and κ (log-scale).
2. **μ–κ relationship** — scatter of μ vs log₁₀ κ, with medians and outlier labels.
3. **κ calibration trace** — iteration vs empirical predictive coverage, target line overlay.
4. **ΔLL per mutation** — Beta–Binomial dynamic vs pooled Binomial.
5. **PIT (mid‑P) histogram** — uniform check.
6. **PIT KS‑p by strata** — heatmap (coverage × time).
7. **Empirical coverage by strata** — heatmap centered on the target coverage.
8. **Z‑score diagnostics** — histogram with N(0,1) overlay and QQ plot.
9. **Mutation panel (multi‑site)** — observed AF, predicted μ(t), and exact Beta–Binomial predictive bands.
10. **Global vs site trends** — μ(t) lines for the global trend and top sites for a chosen mutation.
11. **Process noise summary (optional)** — distributions of tuned qₗₗ and q_b if present.
12. **Sampling cadence** — n time points vs Δt summaries per (site, mutation).

The notebook also writes a multi‑page PDF to `results/priors/figs/priors_diagnostics_pack.pdf`.

**Tip:** Place this notebook inside your repository (e.g., `notebooks/`) and run it from there. It resolves paths relative to the notebook’s parent directory.

In [ ]:
# Imports & configuration ---------------------------------------------------
import os
from pathlib import Path
import math
import numpy as np
import pandas as pd

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.backends.backend_pdf import PdfPages

from scipy.stats import betabinom, norm

# --- Resolve repo root one level above this notebook -----------------------
NOTEBOOK_DIR = Path(__file__).resolve().parent if "__file__" in globals() else Path(os.getcwd()).resolve()
REPO_ROOT = NOTEBOOK_DIR.parent
print(f"[INFO] Using repository root: {REPO_ROOT}")

# Base directory for pipeline outputs
TABLES_DIR = REPO_ROOT / "results" / "priors" / "tables"

# Where to save figures/reports
FIG_DIR = REPO_ROOT / "results" / "priors" / "figs"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Constants
GLOBAL_SITE_ID = "__GLOBAL__"
PRED_INTERVAL_Q = 0.80  # central predictive interval for y (counts) / AF

def set_matplotlib_style():
    \"\"\"Modern, publication-friendly Matplotlib-only style.\"\"\"
    mpl.rcParams.update({
        "figure.dpi": 140,
        "savefig.dpi": 140,
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "axes.grid": True,
        "grid.alpha": 0.18,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "legend.frameon": False,
        "legend.borderaxespad": 0.6,
        "font.size": 10,
        "axes.titlesize": 12,
        "axes.labelsize": 11,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
    })

set_matplotlib_style()

# Helper: parse dates on read when present
def _read_csv(name: str, base: Path = TABLES_DIR) -> pd.DataFrame:
    p = base / name
    if not p.exists():
        print(f"[WARN] Missing file: {p}")
        return pd.DataFrame()
    df = pd.read_csv(p)
    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"], errors="coerce")
    return df

print(f"Reading tables from: {TABLES_DIR.resolve()}")
print(f"Saving figures to:  {FIG_DIR.resolve()}")

## Load tables

In [ ]:
pri_hyp   = _read_csv("priors_hyperparams.csv")
tl        = _read_csv("priors_time_local.csv")
residuals = _read_csv("residuals.csv")
pit_df    = _read_csv("prior_predictive_pit.csv")
fitll     = _read_csv("fit_ll.csv")
gates     = _read_csv("gates_summary.csv")
cov_str   = _read_csv("coverage_by_strata.csv")
pit_ks    = _read_csv("pit_ks_by_strata.csv")
mle       = _read_csv("mle_per_mutation.csv")
mle_curv  = _read_csv("mle_curvature.csv")
eb_pop    = _read_csv("eb_population_prior.csv")
proc_noise= _read_csv("process_noise_estimates.csv")
cal_trace = _read_csv("kappa_calibration_trace.csv")
ser_index = _read_csv("smoothing_series_index.csv")

def _brief(df: pd.DataFrame, name: str):
    if df.empty:
        print(f"[{name}] empty")
        return
    cols = ", ".join(list(df.columns)[:6] + (["..."] if df.shape[1] > 6 else []))
    print(f"[{name}] shape={df.shape}; cols=[{cols}]")

_brief(pri_hyp,   "priors_hyperparams")
_brief(tl,        "priors_time_local")
_brief(residuals, "residuals")
_brief(pit_df,    "prior_predictive_pit")
_brief(fitll,     "fit_ll")
_brief(gates,     "gates_summary")
_brief(cov_str,   "coverage_by_strata")
_brief(pit_ks,    "pit_ks_by_strata")
_brief(mle,       "mle_per_mutation")
_brief(mle_curv,  "mle_curvature")
_brief(eb_pop,    "eb_population_prior")
_brief(proc_noise,"process_noise_estimates")
_brief(cal_trace, "kappa_calibration_trace")
_brief(ser_index, "smoothing_series_index")

## Utilities

In [ ]:
def predictive_af_interval(n: np.ndarray, mu: np.ndarray, kappa: np.ndarray, q: float = PRED_INTERVAL_Q):
    \"\"\"Equal-tailed predictive interval for AF using Beta–Binomial quantiles.\"\"\"
    n = np.asarray(n, int)
    mu = np.asarray(mu, float)
    kappa = np.asarray(kappa, float)
    a = np.clip(mu, 1e-12, 1-1e-12) * np.clip(kappa, 1e-9, 1e12)
    b = (1.0 - np.clip(mu, 1e-12, 1-1e-12)) * np.clip(kappa, 1e-9, 1e12)
    alpha = (1.0 - q) / 2.0
    y_lo = betabinom.ppf(alpha, n, a, b).astype(float)
    y_hi = betabinom.ppf(1.0 - alpha, n, a, b).astype(float)
    with np.errstate(invalid="ignore", divide="ignore"):
        lo = np.where(n > 0, y_lo / n, np.nan)
        hi = np.where(n > 0, y_hi / n, np.nan)
    return lo, hi

def format_date_axis(ax, rotation=0, ha="center"):
    ax.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=4, maxticks=10))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
    for lbl in ax.get_xticklabels():
        lbl.set_rotation(rotation)
        lbl.set_horizontalalignment(ha)

def top_mutations_by_rows(df: pd.DataFrame, k: int = 1) -> list:
    if df.empty: return []
    vc = df["mutation"].value_counts().sort_values(ascending=False)
    return list(vc.index[:k])

def top_sites_for_mutation(res: pd.DataFrame, mutation: str, k: int = 6) -> list:
    sub = res[res["mutation"] == mutation]
    if sub.empty: return []
    vc = sub["site_id"].value_counts().sort_values(ascending=False)
    return list(vc.index[:k])

def _nan_guard(x):
    return x[~np.isnan(x)]

## P1 — Priors distributions (μ and κ)

In [ ]:
def fig_priors_distributions(pri_hyp: pd.DataFrame):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    if not pri_hyp.empty:
        mu = pri_hyp["mu"].to_numpy(float)
        kap = np.clip(pri_hyp["kappa"].to_numpy(float), 1e-12, 1e12)
        axes[0].hist(mu, bins=50, alpha=0.85)
        axes[0].axvline(np.nanmedian(mu), ls="--", lw=1.4, label=f"median={np.nanmedian(mu):.3f}")
        axes[0].set_title("Prior μ distribution")
        axes[0].set_xlabel("μ"); axes[0].set_ylabel("Count"); axes[0].legend(loc="best")

        axes[1].hist(np.log10(kap), bins=50, alpha=0.85)
        axes[1].axvline(np.log10(np.nanmedian(kap)), ls="--", lw=1.4, label=f"median log10 κ={np.log10(np.nanmedian(kap)):.2f}")
        axes[1].set_title("Prior κ distribution (log10 scale)")
        axes[1].set_xlabel("log10 κ"); axes[1].set_ylabel("Count"); axes[1].legend(loc="best")
    else:
        axes[0].text(0.5, 0.5, "priors_hyperparams.csv missing", ha="center")
        axes[1].axis("off")
    fig.suptitle("P1 — Priors distributions", y=1.02)
    fig.tight_layout()
    return fig

fig1 = fig_priors_distributions(pri_hyp); fig1

## P2 — μ vs κ (log₁₀) scatter with medians and outlier labels

In [ ]:
def fig_mu_vs_kappa(pri_hyp: pd.DataFrame, label_outliers: int = 6):
    fig, ax = plt.subplots(figsize=(7.5, 6))
    if pri_hyp.empty:
        ax.text(0.5, 0.5, "priors_hyperparams.csv missing", ha="center"); return fig
    mu = pri_hyp["mu"].to_numpy(float)
    kap = np.clip(pri_hyp["kappa"].to_numpy(float), 1e-12, 1e12)
    x = mu; y = np.log10(kap)
    ax.scatter(x, y, s=9, alpha=0.6)
    ax.axvline(np.nanmedian(x), ls="--", lw=1.0, color="gray")
    ax.axhline(np.nanmedian(y), ls="--", lw=1.0, color="gray")
    ax.set_xlabel("μ"); ax.set_ylabel("log10 κ"); ax.set_title("P2 — μ vs log10 κ")
    # Label outliers by extreme κ
    if label_outliers and not pri_hyp["mutation"].isna().all():
        idx_hi = np.argsort(y)[-label_outliers:]
        idx_lo = np.argsort(y)[:label_outliers]
        for idx in np.unique(np.concatenate([idx_hi, idx_lo])):
            mut = str(pri_hyp.iloc[idx]["mutation"])
            ax.annotate(mut, (x[idx], y[idx]), fontsize=7, xytext=(3,3), textcoords="offset points")
    fig.tight_layout(); return fig

fig2 = fig_mu_vs_kappa(pri_hyp); fig2

## P3 — κ calibration trace (coverage vs iteration, target band)

In [ ]:
def fig_kappa_calibration(cal_trace: pd.DataFrame, target_q: float = PRED_INTERVAL_Q):
    fig, ax = plt.subplots(figsize=(7.2, 4.8))
    if cal_trace.empty or "coverage_emp" not in cal_trace.columns:
        ax.text(0.5, 0.5, "kappa_calibration_trace.csv missing", ha="center"); return fig
    it = cal_trace["iter"].to_numpy(int)
    cov = cal_trace["coverage_emp"].to_numpy(float)
    ax.plot(it, cov, marker="o", lw=2, label="empirical coverage")
    ax.axhline(target_q, color="black", lw=1.4, ls="--", label=f"target={target_q:.2f}")
    ax.set_xlabel("iteration"); ax.set_ylabel("empirical coverage")
    ax.set_ylim(0.0, 1.0); ax.set_title("P3 — κ calibration (exact mid‑P)")
    ax.legend(loc="best"); fig.tight_layout(); return fig

fig3 = fig_kappa_calibration(cal_trace); fig3

## P4 — ΔLL per mutation (BB dynamic − Binomial pooled) — top 30 by |ΔLL|

In [ ]:
def fig_delta_ll(fitll: pd.DataFrame, top: int = 30):
    fig, ax = plt.subplots(figsize=(10, 6))
    if fitll.empty or "delta_ll" not in fitll.columns:
        ax.text(0.5, 0.5, "fit_ll.csv missing", ha="center"); return fig
    srt = fitll.dropna(subset=["delta_ll"]).sort_values("delta_ll", ascending=False)
    top_df = pd.concat([srt.head(top//2), srt.tail(top//2)], axis=0)
    ylab = list(top_df["mutation"])
    vals = list(top_df["delta_ll"])
    ax.barh(ylab[::-1], vals[::-1], alpha=0.85)
    ax.axvline(0.0, color="black", lw=1.0)
    ax.set_xlabel("ΔLL (weighted)"); ax.set_title("P4 — ΔLL per mutation (top & bottom)")
    ax.set_yticks(np.arange(len(ylab))); ax.set_yticklabels(ylab, fontsize=7)
    fig.tight_layout(); return fig

fig4 = fig_delta_ll(fitll); fig4

## P5 — PIT (mid‑P) histogram with uniform baseline

In [ ]:
def fig_pit_hist(pit_df: pd.DataFrame, bins: int = 40):
    fig, ax = plt.subplots(figsize=(7.2, 4.8))
    if pit_df.empty or "pit_mid" not in pit_df.columns:
        ax.text(0.5, 0.5, "prior_predictive_pit.csv missing", ha="center"); return fig
    x = pit_df["pit_mid"].to_numpy(float)
    x = x[np.isfinite(x) & (x >= 0.0) & (x <= 1.0)]
    ax.hist(x, bins=bins, density=True, alpha=0.85)
    ax.plot([0, 1], [1, 1], color="black", lw=1.4, ls="--", label="Uniform(0,1)")
    ax.set_xlim(0, 1); ax.set_ylim(0, None)
    ax.set_xlabel("PIT (mid‑P)"); ax.set_ylabel("Density"); ax.set_title("P5 — PIT histogram")
    ax.legend(loc="best"); fig.tight_layout(); return fig

fig5 = fig_pit_hist(pit_df); fig5

## P6 — PIT KS‑p by strata (coverage × time) — heatmap

In [ ]:
def fig_pit_ks_heatmap(pit_ks: pd.DataFrame):
    fig, ax = plt.subplots(figsize=(8.5, 5.8))
    if pit_ks.empty or not {"cov_bin","time_bin","ks_p"} <= set(pit_ks.columns):
        ax.text(0.5, 0.5, "pit_ks_by_strata.csv missing", ha="center"); return fig
    piv = pit_ks.pivot_table(index="cov_bin", columns="time_bin", values="ks_p", aggfunc="mean")
    im = ax.imshow(piv.values, aspect="auto", interpolation="nearest", vmin=0.0, vmax=1.0)
    ax.set_title("P6 — PIT KS‑p by strata"); ax.set_xlabel("time_bin"); ax.set_ylabel("cov_bin")
    ax.set_xticks(np.arange(piv.shape[1])); ax.set_xticklabels(piv.columns)
    ax.set_yticks(np.arange(piv.shape[0])); ax.set_yticklabels(piv.index)
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04); cbar.set_label("KS p‑value")
    fig.tight_layout(); return fig

fig6 = fig_pit_ks_heatmap(pit_ks); fig6

## P7 — Empirical coverage by strata (coverage × time) — heatmap

In [ ]:
def fig_coverage_heatmap(cov_str: pd.DataFrame, target_q: float = PRED_INTERVAL_Q, tol: float = 0.02):
    fig, ax = plt.subplots(figsize=(8.5, 5.8))
    if cov_str.empty or not {"cov_bin","time_bin","coverage_emp"} <= set(cov_str.columns):
        ax.text(0.5, 0.5, "coverage_by_strata.csv missing", ha="center"); return fig
    piv = cov_str.pivot_table(index="cov_bin", columns="time_bin", values="coverage_emp", aggfunc="mean")
    im = ax.imshow(piv.values, aspect="auto", interpolation="nearest", vmin=0.0, vmax=1.0, cmap="viridis")
    ax.set_title("P7 — Empirical coverage by strata"); ax.set_xlabel("time_bin"); ax.set_ylabel("cov_bin")
    ax.set_xticks(np.arange(piv.shape[1])); ax.set_xticklabels(piv.columns)
    ax.set_yticks(np.arange(piv.shape[0])); ax.set_yticklabels(piv.index)
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04); cbar.set_label("coverage")
    # overlay target band
    ax2 = ax.twinx(); ax2.set_yticks([])
    ax.axhline(-0.5, color="white", lw=1.0)  # cosmetic
    fig.tight_layout(); return fig

fig7 = fig_coverage_heatmap(cov_str); fig7

## P8 — Z‑scores: histogram with N(0,1) overlay and QQ plot

In [ ]:
def fig_z_diagnostics(residuals: pd.DataFrame):
    fig = plt.figure(figsize=(12, 4.8))
    gs = fig.add_gridspec(1, 2, wspace=0.25)
    ax1 = fig.add_subplot(gs[0, 0])
    ax2 = fig.add_subplot(gs[0, 1])
    if residuals.empty or "resid" not in residuals.columns:
        ax1.text(0.5, 0.5, "residuals.csv missing", ha="center"); ax2.axis("off"); return fig
    z = residuals["resid"].to_numpy(float)
    z = z[np.isfinite(z)]
    if z.size == 0:
        ax1.text(0.5, 0.5, "no finite residuals", ha="center"); ax2.axis("off"); return fig
    bins = min(60, max(30, int(np.sqrt(z.size))))
    ax1.hist(z, bins=bins, density=True, alpha=0.85, label="z")
    xs = np.linspace(-4, 4, 400); ax1.plot(xs, norm.pdf(xs), color="black", lw=1.4, label="N(0,1)")
    ax1.set_title("Z histogram"); ax1.set_xlabel("z"); ax1.set_ylabel("density"); ax1.legend(loc="best")

    z_sorted = np.sort(z); ppos = (np.arange(1, z_sorted.size + 1) - 0.5) / z_sorted.size
    q_theo = norm.ppf(ppos)
    ax2.plot(q_theo, z_sorted, marker="o", ms=2.0, linestyle="none", alpha=0.5)
    lim = 1.05 * np.nanmax(np.abs(np.concatenate([q_theo, z_sorted])))
    lim = max(lim, 1.0)
    ax2.plot([-lim, lim], [-lim, lim], color="gray", lw=1.2)
    ax2.set_xlim(-lim, lim); ax2.set_ylim(-lim, lim)
    ax2.set_title("Z QQ"); ax2.set_xlabel("theoretical"); ax2.set_ylabel("empirical")
    fig.suptitle("P8 — Z‑score diagnostics", y=1.02); fig.tight_layout(); return fig

fig8 = fig_z_diagnostics(residuals); fig8

## P9 — Mutation panel: observed AF, predicted μ(t), predictive bands (multi‑site)

In [ ]:
# Choose a mutation automatically if not specified
MUTATION_TO_PLOT = None  # e.g., "S:Q498R" — set to None to auto-pick most frequent

def fig_mutation_multisite_panel(residuals: pd.DataFrame, mutation: str = None, sites_max: int = 6):
    fig = plt.figure(figsize=(14, 9))
    if residuals.empty:
        plt.text(0.5, 0.5, "residuals.csv missing", ha="center"); return fig
    if mutation is None:
        muts = top_mutations_by_rows(residuals, k=1)
        mutation = muts[0] if muts else None
    if mutation is None:
        plt.text(0.5, 0.5, "no mutations available", ha="center"); return fig
    sub = residuals[residuals["mutation"] == mutation].copy()
    if sub.empty:
        plt.text(0.5, 0.5, f"no rows for {mutation}", ha="center"); return fig
    sites = top_sites_for_mutation(sub, mutation, k=sites_max)
    nrows = int(math.ceil(len(sites)/3)); ncols = 3
    gs = fig.add_gridspec(nrows, ncols, wspace=0.25, hspace=0.35)
    for i, site in enumerate(sites):
        ax = fig.add_subplot(gs[i // ncols, i % ncols])
        s = sub[sub["site_id"] == site].sort_values("date")
        if s.empty:
            ax.axis("off"); continue
        dt = pd.to_datetime(s["date"])
        n = s["coverage"].to_numpy(int)
        mu = s["mu_t"].fillna(s.get("af", np.nan)).to_numpy(float)
        kap = s.get("kappa", pd.Series(np.nan, index=s.index)).to_numpy(float)
        lo, hi = predictive_af_interval(n, mu, kap, PRED_INTERVAL_Q)
        # Observed AF scatter
        with np.errstate(invalid="ignore", divide="ignore"):
            af_obs = np.where(n>0, s["count"].to_numpy(float)/n, np.nan)
        ax.scatter(dt, af_obs, s=10, alpha=0.5, label="obs AF")
        # Predicted μ(t) + band
        ax.plot(dt, mu, lw=2, label="μ(t)")
        ax.fill_between(dt, lo, hi, alpha=0.2, label=f"{int(PRED_INTERVAL_Q*100)}% pred band")
        ax.set_ylim(-0.02, 1.02)
        ax.set_title(f"{site}")
        format_date_axis(ax, rotation=45, ha="right")
        if i == 0:
            ax.legend(loc="upper left", ncol=3, bbox_to_anchor=(0, 1.25))
    fig.suptitle(f"P9 — {mutation}: multi‑site panel", y=1.02)
    fig.tight_layout()
    return fig

fig9 = fig_mutation_multisite_panel(residuals, MUTATION_TO_PLOT, sites_max=6); fig9

## P10 — Global vs site trends (μ(t)) for a selected mutation

In [ ]:
def fig_global_vs_sites(tl: pd.DataFrame, mutation: str = None, sites_max: int = 8):
    fig, ax = plt.subplots(figsize=(12, 6))
    if tl.empty:
        ax.text(0.5, 0.5, "priors_time_local.csv missing", ha="center"); return fig
    if mutation is None:
        muts = top_mutations_by_rows(tl, k=1)
        mutation = muts[0] if muts else None
    if mutation is None:
        ax.text(0.5, 0.5, "no mutations available", ha="center"); return fig
    t = tl[tl["mutation"] == mutation].copy()
    if t.empty:
        ax.text(0.5, 0.5, f"no rows for {mutation}", ha="center"); return fig
    # Global line
    g = t[t["site_id"] == GLOBAL_SITE_ID].sort_values("date")
    if not g.empty:
        ax.plot(g["date"], g["mu_t"], lw=3.0, label="GLOBAL μ(t)")
    # Top sites
    sites = (t[t["site_id"] != GLOBAL_SITE_ID]["site_id"].value_counts()
             .sort_values(ascending=False).index[:sites_max])
    for s in sites:
        loc = t[(t["site_id"] == s)].sort_values("date")
        ax.plot(loc["date"], loc["mu_t"], lw=1.5, alpha=0.8, label=str(s))
    ax.set_ylim(-0.02, 1.02); ax.set_title(f"P10 — μ(t) global vs top sites for {mutation}")
    ax.set_xlabel("date"); ax.set_ylabel("μ(t)")
    format_date_axis(ax, rotation=45, ha="right")
    ax.legend(loc="upper left", ncol=4, bbox_to_anchor=(0, 1.25))
    fig.tight_layout(); return fig

fig10 = fig_global_vs_sites(tl, mutation=None, sites_max=8); fig10

## P11 — Process noise summary (if tuned per mutation)

In [ ]:
def fig_process_noise(proc_noise: pd.DataFrame):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    if proc_noise.empty:
        for ax in axes: ax.text(0.5, 0.5, "process_noise_estimates.csv missing", ha="center")
        return fig
    if "q_a_hat" in proc_noise.columns and "q_b_hat" in proc_noise.columns and proc_noise.shape[0] == 1:
        qa = float(proc_noise.iloc[0]["q_a_hat"]); qb = float(proc_noise.iloc[0]["q_b_hat"])
        axes[0].text(0.5, 0.6, f"q_LL (global median) = {qa:g}", ha="center")
        axes[0].axis("off")
        axes[1].text(0.5, 0.6, f"q_b (global median) = {qb:g}", ha="center")
        axes[1].axis("off")
    else:
        if "q_LL_hat" in proc_noise.columns:
            axes[0].hist(proc_noise["q_LL_hat"].to_numpy(float), bins=40, alpha=0.85)
            axes[0].set_title("q_LL per mutation")
        if "q_b_hat" in proc_noise.columns:
            axes[1].hist(proc_noise["q_b_hat"].to_numpy(float), bins=40, alpha=0.85)
            axes[1].set_title("q_b per mutation")
    fig.suptitle("P11 — Process noise", y=1.02); fig.tight_layout(); return fig

fig11 = fig_process_noise(proc_noise); fig11

## P12 — Sampling cadence: n time points vs Δt summaries

In [ ]:
def fig_sampling_cadence(ser_index: pd.DataFrame):
    fig, ax = plt.subplots(figsize=(8.5, 5.8))
    if ser_index.empty or "n_timepoints" not in ser_index.columns:
        ax.text(0.5, 0.5, "smoothing_series_index.csv missing", ha="center"); return fig
    x = ser_index["n_timepoints"].to_numpy(float)
    y = ser_index.get("dt_med", pd.Series(np.nan, index=ser_index.index)).to_numpy(float)
    ax.scatter(x, y, s=12, alpha=0.6)
    ax.set_xlabel("n_timepoints per (site, mutation)")
    ax.set_ylabel("median Δt (days)")
    ax.set_title("P12 — Sampling cadence")
    fig.tight_layout(); return fig

fig12 = fig_sampling_cadence(ser_index); fig12

## Export: multi‑page PDF

In [ ]:
pdf_path = FIG_DIR / "priors_diagnostics_pack.pdf"
figs = [fig1, fig2, fig3, fig4, fig5, fig6, fig7, fig8, fig9, fig10, fig11, fig12]
with PdfPages(pdf_path) as pdf:
    for i, f in enumerate(figs, 1):
        try:
            pdf.savefig(f, bbox_inches="tight")
        except Exception:
            pass
print(f"[OK] Wrote multi‑page PDF to: {pdf_path}")